# Comparison of Coupling Methods on 8gaussian -> Moon

This notebook compares three different flow matching coupling methods:
1. **Independent-CFM**
2. **Entropic-CFM**
3. **OT-CFM**

It uses a single shared set of initial points $x_0$ and target points $x_1$ to ensure a fair comparison.

In [ ]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from torchdyn.core import NeuralODE

from guided_flow.backbone.mlp import MLP
from guided_flow.backbone.wrapper import MLPWrapper
from guided_flow.distributions.base import get_distribution
from guided_flow.utils.misc import deterministic, get_device

In [ ]:
# Helper to get the latest trained model
def get_latest_model(base_dir, prefix):
    pattern = os.path.join(base_dir, f"{prefix}*.pth")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"Could not find any models matching: {pattern}")
    # Sort files by modification time descending
    sorted_files = sorted(files, key=os.path.getmtime, reverse=True)
    latest_file = sorted_files[0]
    print(f"Loaded latest model: {os.path.basename(latest_file)}")
    return latest_file

In [ ]:
# Setup device and seed
device = get_device()
deterministic(0)

x0_dist = '8gaussian'
x1_dist = 'moon'
MLP_WIDTH = 256

# Sample identical points for fair comparison
batch_size = 512
x0_sampler = get_distribution(x0_dist).sample
x1_sampler = get_distribution(x1_dist).sample

# Sample identical start points and target points
x0_samples = x0_sampler(batch_size=batch_size, device=device)
x1_samples = x1_sampler(batch_size=batch_size, device=device)

In [ ]:
# Function to load and run trajectories
def run_trajectories(cfm_name):
    # Initialize model
    model = MLP(dim=2, w=MLP_WIDTH, time_varying=True).to(device)
    
    # Locate log folder
    log_dir = f"../logs/{x0_dist}-{x1_dist}/{cfm_name}_{x0_dist}_{x1_dist}"
    prefix = f"{cfm_name}_{x0_dist}_{x1_dist}"
    
    # Load state dict
    model_path = get_latest_model(log_dir, prefix)
    model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))
    model.eval()
    
    # Wrap and run ODE
    node = NeuralODE(MLPWrapper(model), solver="euler", sensitivity="adjoint")
    with torch.no_grad():
        # 100 steps from 0 to 1
        traj = node.trajectory(x0_samples, t_span=torch.linspace(0, 1, 100))
    
    return traj.cpu()

In [ ]:
print("Simulating Independent-CFM...")
traj_cfm = run_trajectories("cfm")

print("Simulating OT-CFM...")
traj_ot = run_trajectories("ot_cfm")

print("Simulating Entropic-CFM...")
traj_entropic = run_trajectories("entropic_ot_cfm")

In [ ]:
# Plotting the three coupling methods in one figure, in one row
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
methods = [
    ("Independent-CFM", traj_cfm),
    ("Entropic-CFM", traj_entropic),
    ("OT-CFM", traj_ot)
]

for idx, (name, traj) in enumerate(methods):
    ax = axs[idx]
    
    # 1. Plot the actual trajectory endpoints (blue dots)
    ax.scatter(traj[-1, :, 0], traj[-1, :, 1], color='blue', alpha=0.5, s=8, label='Target (Moons)' if idx == 0 else "", zorder=1)
    
    # 2. Plot trajectories as thin semi-transparent lines
    T, B, _ = traj.shape
    for i in range(B):
        ax.plot(traj[:, i, 0], traj[:, i, 1], color='red', alpha=0.35, linewidth=1.2, zorder=2)
        
    # 3. Plot the starting points (black dots)
    ax.scatter(traj[0, :, 0], traj[0, :, 1], color='black', alpha=0.8, s=10, zorder=3, label='Source (8 Gaussian)' if idx == 0 else "")
    
    ax.set_title(name, fontsize=16, fontweight='bold')
    ax.set_xlim(-2.0, 2.0)
    ax.set_ylim(-2.0, 2.0)
    ax.set_aspect('equal')
    ax.axis('off')
    
    if idx == 0:
        ax.legend(loc='lower left', frameon=False, fontsize=12)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/coupling_comparison.svg", format="svg", bbox_inches="tight")
plt.savefig("figures/coupling_comparison.png", format="png", dpi=300, bbox_inches="tight")
plt.show()